# Colab GPU training: writer-ID + CRNN at full caps, plus a VATr smoke test

This notebook runs the ROADMAP steps this laptop can't: it has no CUDA, 2 CPU cores and 8 GB of RAM. It does four things:

1. Clones `SUDER14/Handwritting` (branch `master`) and rebuilds `data/iam/` from Hugging Face (Teklia/IAM-line + `forms.txt`).
2. Trains the **writer-ID** net (Task 1.1, **full cap: 40 epochs**) and the **CRNN** legibility recognizer (Task 1.3, **full cap: 60 epochs**) on the GPU. Early stopping on the validation metric is kept, as in the ROADMAP. All other hyperparameters come from `config/config.yaml`, the same as the CPU runs.
3. Clones the official VATr repo at the pinned SHA, downloads its released checkpoint, and runs a **one-writer, K=1 VATr inference smoke test** (`scripts/vatr_smoke.py`). An optional cell runs the Task 3.2 reproduction (published HWD 0.828).
4. Copies everything you need to bring home into Google Drive.

**Before you start:** Runtime → Change runtime type → **GPU**. Put `forms.txt` in Drive at `MyDrive/handwriting/forms.txt`, or upload it when cell 3 asks. It's gitignored because IAM is licensed.

## Run IDs

Checkpoints are written the same way as the local runs, `models/checkpoints/<run_id>/`, with `--run-name gpu_colab_r1`:

| task | local CPU baseline (superseded) | this notebook |
|---|---|---|
| writer-ID | `writerid_20260921T180911Z_ed897d02_cpu_r1` | `writerid_<utc>_<sha>_gpu_colab_r1` |
| CRNN | `crnn_20260922T003942Z_d1f87b49_cpu_r1` | `crnn_<utc>_<sha>_gpu_colab_r1` |

Each run directory holds `config.json` (records `device` + GPU name), `train_log.csv`, `best.pt`, `last.pt`, `run.json`, and `test_metrics.json`.

## What to download at the end, and where it goes locally

The last cell copies these into `MyDrive/handwriting/colab_out/`. Download that folder:

| from Drive `colab_out/` | put it locally at |
|---|---|
| `checkpoints/writerid_*_gpu_colab_r1/` (whole dir) | `models/checkpoints/writerid_*_gpu_colab_r1/` |
| `checkpoints/crnn_*_gpu_colab_r1/` (whole dir) | `models/checkpoints/crnn_*_gpu_colab_r1/` |
| `vatr_smoke/` | `results/vatr_smoke_colab/` |
| `vatr_reproduction.json` (optional cell) | `results/vatr_reproduction.json` |
| `logs/*.log` | `logs/` |

Copy the run directories **as they are, and don't rename them**: the run_id is written inside `config.json` and `run.json`.

**Important, for Claude on the local side:** the protocol picks the *newest* writer-ID and CRNN runs by name (`evaluation.writer_id_run` / `evaluation.crnn_run: null` in `config.yaml`). The GPU runs sort after the CPU ones, so dropping them in **silently changes the measurement instruments** for every later protocol row. After they arrive, pin both keys explicitly in `config.yaml`, log the switch in ROADMAP_STATE.md, and re-score the existing rows (real/baseline/glyph_vae) with the new instruments before comparing them with VATr rows. Otherwise the table would mix instruments.

## 1. GPU check

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")

## 2. Clone the repo and install requirements
Colab's preinstalled CUDA build of torch/torchvision is kept. Reinstalling torch from `requirements.txt` could swap it for a mismatched build.

In [ ]:
REPO_URL = "https://github.com/SUDER14/Handwritting.git"
BRANCH = "master"
ROOT = "/content/Handwritting"
import os
if not os.path.exists(ROOT):
    !git clone -b {BRANCH} {REPO_URL} {ROOT}
%cd {ROOT}
!git log --oneline -1
!grep -v -E '^(torch|torchvision)\b' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt gdown
!mkdir -p logs

## 3. `forms.txt` (licensed IAM metadata, not in git)
This cell looks in Drive at `MyDrive/handwriting/forms.txt` first. If it isn't there, it asks you to upload the file.

In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/handwriting"
os.makedirs(DRIVE_DIR, exist_ok=True)
if os.path.exists(f"{DRIVE_DIR}/forms.txt"):
    !cp "{DRIVE_DIR}/forms.txt" forms.txt
elif not os.path.exists("forms.txt"):
    up = files.upload()                      # choose forms.txt from your machine
    assert "forms.txt" in up, "upload the file named forms.txt"
!head -c 300 forms.txt; echo; wc -l forms.txt

## 4. Rebuild `data/iam/` from Hugging Face
`join_teklia.py` attributes each Teklia line to an IAM writer and form (it writes `data/processed/teklia_join/retained.csv`). `materialize_iam.py` then writes `data/iam/ascii/{forms,lines}.txt` and `data/iam/lines/**.png` (about 900 MB). The writer split files (`data/splits/vatr/*.json`) are in git.

In [ ]:
!python scripts/join_teklia.py --forms forms.txt 2>&1 | tee logs/colab_join_teklia.log | tail -20
!python scripts/materialize_iam.py 2>&1 | tee logs/colab_materialize_iam.log | tail -10
!find data/iam/lines -name '*.png' | wc -l

## 5. Sanity checks before training
These are the leakage test and the split check. Training also asserts that no test writer is in train/val.

In [ ]:
!python -m pytest -q -k "leak or split" 2>&1 | tail -5
!python scripts/check_split.py 2>&1 | tail -15

## 6. Writer-ID, full cap 40 epochs (Task 1.1)
It keeps early stopping on val retrieval (patience from config). At the end it evaluates the TEST writers once and writes `test_metrics.json` (GATE 1 needs top-1 ≥ 0.60).

If Colab disconnects, rerun this cell. It creates a *new* run dir and never overwrites the old one. Just make sure you download the complete one.

In [ ]:
!python scripts/train_writer_id.py --epochs 40 --run-name gpu_colab_r1 --device cuda 2>&1 | tee logs/writerid_gpu_colab_r1.log | grep -v -E '^\s*$' | tail -60
!ls -d models/checkpoints/writerid_*_gpu_colab_r1 && cat models/checkpoints/writerid_*_gpu_colab_r1/test_metrics.json

## 7. CRNN legibility recognizer, full cap 60 epochs (Task 1.3)
It's frozen after training. GATE 1 needs test CER ≤ 0.15. `crnn.max_line_width` stays at the config value, so the test-line set is identical to the CPU run's.

In [ ]:
!python scripts/train_crnn.py --epochs 60 --run-name gpu_colab_r1 --device cuda 2>&1 | tee logs/crnn_gpu_colab_r1.log | tail -70
!cat models/checkpoints/crnn_*_gpu_colab_r1/test_metrics.json | head -20

## 8. Official VATr at the pinned SHA + released checkpoint
The released files (`vatr.pth`, `IAM-32.pickle`, `resnet_18_pretrained.pth`) come from the Google Drive folder linked in VATr's README, fetched with `gdown`. If gdown hits a Drive quota error, download the folder by hand from that link and upload the files into `third_party/VATr/files/`.

In [ ]:
VATR_SHA = "183501e02113a3b4b636f2a2c5443771501c8f2b"
if not os.path.exists("third_party/VATr"):
    !git clone https://github.com/aimagelab/VATr.git third_party/VATr
!git -C third_party/VATr checkout -q {VATR_SHA} && git -C third_party/VATr log --oneline -1
if not os.path.exists("third_party/VATr/files/vatr.pth"):
    !cd third_party/VATr/files && gdown --folder "https://drive.google.com/drive/folders/1FGJe2uCuK8T9HrFzY_Zc-KMIo0oPJGGY" -O . --remaining-ok
    # gdown --folder puts files in a sub-folder: move them up next to unifont.pickle
    !cd third_party/VATr/files && find . -mindepth 2 -type f \( -name '*.pth' -o -name '*.pickle' \) -exec mv -n {} . \;
!ls -la third_party/VATr/files | grep -E 'vatr.pth|IAM-32|resnet_18|unifont'

## 9. VATr inference smoke test: one writer, K=1
This runs `scripts/vatr_smoke.py`: the first protocol test writer, one reference line split into word crops (the logged adaptation for missing IAM word boxes, cycled up to VATr's 15 style slots), and the text of one of that writer's other lines. The script exits non-zero if the output is empty or blank.

In [ ]:
!python scripts/vatr_smoke.py --device cuda --out results/vatr_smoke 2>&1 | tail -40
from IPython.display import Image, display
for f in ["reference.png", "generated.png"]:
    print(f); display(Image(f"results/vatr_smoke/{f}"))

## 10. (Optional) Task 3.2: reproduce VATr's published IAM HWD (0.828) with VATr's own generation code
This needs the official HWD repo (pinned SHA) and its VGG16 backbone (about 675 MB, which HWD downloads itself on first use). GATE 3 requires the result to be within 10%, i.e. 0.745 to 0.911.

In [ ]:
HWD_SHA = "eabb5b5ede5899ac7f8b716429cbd1a1c97c7485"
if not os.path.exists("third_party/HWD"):
    !git clone https://github.com/aimagelab/HWD.git third_party/HWD
!git -C third_party/HWD checkout -q {HWD_SHA}
# HWD deps (now also in requirements.txt); VATr's own requirements.txt pins old opencv/wandb and is NOT installed
# (wandb is stubbed at runtime, the rest is already present)
!pip install -q torchmetrics transformers gudhi msgpack
!python scripts/vatr_reproduce.py --device cuda --batch_size 32 2>&1 | tee logs/vatr_reproduce_colab.log | tail -40

## 11. Copy results to Drive (`MyDrive/handwriting/colab_out/`)
This copies only this notebook's GPU runs: `*_gpu_colab_r1` checkpoint dirs, VATr smoke outputs, logs, and the reproduction JSON.

In [ ]:
OUT = f"{DRIVE_DIR}/colab_out"
!mkdir -p "{OUT}/checkpoints" "{OUT}/logs"
!cp -r models/checkpoints/*_gpu_colab_r1 "{OUT}/checkpoints/"
!cp -r results/vatr_smoke "{OUT}/" 2>/dev/null || true
!cp results/vatr_reproduction.json "{OUT}/" 2>/dev/null || true
!cp logs/*.log "{OUT}/logs/"
!git rev-parse HEAD > "{OUT}/git_sha.txt"
!du -sh "{OUT}"/* ; ls -R "{OUT}/checkpoints" | head -30